In [6]:
import pandas as pd
import datetime
import os

# --- 檔案名稱設定 ---
# 雖然副檔名是 .xls，但我們知道它實際上是 CSV 格式
INPUT_FILENAME = "CFB2_integration_history_20251117.xls"
# 建議將截斷後的檔案儲存為 CSV
OUTPUT_FILENAME = f"CFB2_history_truncated_500_{datetime.date.today().strftime('%Y%m%d')}.csv" 

# --- 1. 載入檔案 (修正錯誤：改用 read_csv) ---
try:
    # 根據錯誤訊息，確認檔案內容為 CSV 格式
    df_history = pd.read_csv(INPUT_FILENAME) 
    print(f"✅ 成功載入歷史檔案：{INPUT_FILENAME} (已識別為 CSV 格式)")
except FileNotFoundError:
    print(f"❌ 找不到檔案：{INPUT_FILENAME}。請確認檔案與程式碼在同一個路徑下。")
    exit()
except Exception as e:
    print(f"❌ 讀取檔案時發生無法預期的錯誤: {e}")
    print("💡 如果問題仍存在，請確認檔案 'CFB2_integration_history_20251117.xls' 的內容。")
    exit()


# --- 2. 執行截斷：移除前 500 步 (index 0 到 499) ---
# df_truncated 包含第 501 步到最後的所有資料
# 這是您想要用來分析/生成報告的數據
df_truncated = df_history.iloc[600:].copy()

# --- 3. 儲存截斷後的資料為新的 CSV 歷史檔案 ---
df_truncated.to_csv(OUTPUT_FILENAME, index=False)

print(f"---")
print(f"原始資料總共有 {len(df_history)} 筆記錄。")
print(f"截斷後的資料有 {len(df_truncated)} 筆記錄（已移除前 500 步）。")
print(f"📁 新的歷史數據檔案已儲存至: {OUTPUT_FILENAME}")

✅ 成功載入歷史檔案：CFB2_integration_history_20251117.xls (已識別為 CSV 格式)
---
原始資料總共有 1160 筆記錄。
截斷後的資料有 560 筆記錄（已移除前 500 步）。
📁 新的歷史數據檔案已儲存至: CFB2_history_truncated_500_20251202.csv


In [7]:
df_truncated

,timestamp,cfb_unit,Y1_prediction,Y2_prediction,Y3_prediction
600,2025-11-17 17:21:48.785047,2,0.928327,0.486587,12.835316
601,2025-11-17 17:22:00.579369,2,0.952174,0.474398,13.245167
602,2025-11-17 17:22:10.864419,2,0.951186,0.469413,13.317728
603,2025-11-17 17:22:22.237374,2,0.956544,0.457699,13.394847
604,2025-11-17 17:22:32.581691,2,0.957394,0.441834,13.507626
...,...,...,...,...,...
1155,2025-11-17 18:59:17.760324,2,0.964878,0.565123,9.175912
1156,2025-11-17 18:59:28.838611,2,0.964571,0.570309,9.367255
1157,2025-11-17 18:59:39.641228,2,0.963520,0.581741,9.076211
1158,2025-11-17 18:59:49.955362,2,0.963481,0.580876,9.262645


In [8]:
import pandas as pd

excel_data = pd.read_excel("y2_report_all_tables.xlsx", sheet_name=None)

y2_metrics = excel_data['Y2_預測評估指標_DeSOx_2nd']
y3_metrics = excel_data['Y3_反推評估指標_MLUT4_AT_240']                                                             
detail_data = excel_data['詳細歷史紀錄']
print(detail_data.describe())

              step  last_true_target    actual_y3  prediction_y2  \
count  1160.000000       1158.000000  1160.000000    1150.000000   
mean    580.500000          0.296102    10.538966       0.313547   
std     335.007463          1.232446     3.086076       1.206739   
min       1.000000         -4.741900     0.000000      -4.300700   
25%     290.750000          0.479025     8.700000       0.488100   
50%     580.500000          0.593500    10.700000       0.585200   
75%     870.250000          0.980175    12.900000       0.980100   
max    1160.000000          1.000000    17.800000       1.588300   

       prediction_y3        error    threshold  actual_target  
count    1150.000000  1147.000000  1147.000000    1158.000000  
mean       14.369561     0.054013     0.071077       0.296102  
std        58.185142     0.189589     0.192734       1.232446  
min      -392.182400     0.000000     0.000100      -4.741900  
25%         8.532225     0.002600     0.004000       0.479025  
50%

In [9]:
df_truncated = df_truncated.join(detail_data, how='left')
df_truncated

,timestamp,cfb_unit,Y1_prediction,Y2_prediction,Y3_prediction,step,rebuild_triggered,last_true_target,actual_y3,features_used,prediction_y2,prediction_y3,error,threshold,actual_target
600,2025-11-17 17:21:48.785047,2,0.928327,0.486587,12.835316,601,False,0.4257,14.1,"{'MLUT4_FT-957': 2962.0, 'MLUT4_FT-956': 2808....",0.4866,12.8353,0.0596,0.0948,0.4360
601,2025-11-17 17:22:00.579369,2,0.952174,0.474398,13.245167,602,False,0.4360,14.6,"{'MLUT4_FT-957': 2981.0, 'MLUT4_FT-956': 2813....",0.4744,13.2452,0.0506,0.0909,0.4206
602,2025-11-17 17:22:10.864419,2,0.951186,0.469413,13.317728,603,False,0.4206,15.1,"{'MLUT4_FT-957': 2966.0, 'MLUT4_FT-956': 2808....",0.4694,13.3177,0.0538,0.0870,0.3984
603,2025-11-17 17:22:22.237374,2,0.956544,0.457699,13.394847,604,False,0.3984,15.2,"{'MLUT4_FT-957': 2985.0, 'MLUT4_FT-956': 2813....",0.4577,13.3948,0.0710,0.0877,0.3846
604,2025-11-17 17:22:32.581691,2,0.957394,0.441834,13.507626,605,False,0.3846,14.8,"{'MLUT4_FT-957': 2995.0, 'MLUT4_FT-956': 2788....",0.4418,13.5076,0.0731,0.0859,0.3884
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1155,2025-11-17 18:59:17.760324,2,0.964878,0.565123,9.175912,1156,False,0.5756,8.7,"{'MLUT4_FT-957': 2962.0, 'MLUT4_FT-956': 2813....",0.5651,9.1759,0.0128,0.0156,0.5877
1156,2025-11-17 18:59:28.838611,2,0.964571,0.570309,9.367255,1157,True,0.5877,8.7,"{'MLUT4_FT-957': 2966.0, 'MLUT4_FT-956': 2808....",0.5703,9.3673,0.0226,0.0169,0.6009
1157,2025-11-17 18:59:39.641228,2,0.963520,0.581741,9.076211,1158,False,0.6009,8.7,"{'MLUT4_FT-957': 2957.0, 'MLUT4_FT-956': 2818....",0.5817,9.0762,0.0306,0.0306,0.5991
1158,2025-11-17 18:59:49.955362,2,0.963481,0.580876,9.262645,1159,False,0.5991,8.7,"{'MLUT4_FT-957': 2933.0, 'MLUT4_FT-956': 2798....",0.5809,9.2626,0.0173,0.0293,0.6063


In [10]:
# 假設您已經運行了 CSV 讀取和截斷程式碼，df_truncated 已經在記憶體中

import os
# 匯入您的系統模組
from integration_bridge import CFBIntegrationBridge

# 設定新的報告檔案名稱
NEW_REPORT_FILENAME = "usage_report_truncated_600_y2.html"

# 1. 初始化整合橋接器 (CFB2)
try:
    # **注意：請務必確保 features1.pkl / features2.pkl 的路徑問題已解決**
    bridge = CFBIntegrationBridge(cfb_unit=2) 
    
    # 2. 替換預測器內部的歷史數據
    # 關鍵修正：將 DataFrame 轉換為 List of Dicts，並賦值給 history 屬性
    bridge.predictor_y2.history = df_truncated.to_dict('records') 

    # 3. 呼叫 generate_usage_report 函式，並傳遞正確的參數 report_path
    # 函式會自動使用您在步驟 2 賦值給物件的新數據
    bridge.predictor_y2.generate_usage_report(
        report_path=NEW_REPORT_FILENAME
    ) 

    print(f"---")
    print(f"🎉 使用截斷後的數據生成的新報告已儲存至：{NEW_REPORT_FILENAME}")

except FileNotFoundError:
    print("\n❌ 程式中斷：請確認您已將 features1.pkl 和 features2.pkl 複製到正確的相對路徑。")
except Exception as e:
    print(f"\n❌ 程式中斷：發生未知錯誤: {e}")

⚠️ 無法載入 PI Server 模組，將使用模擬數據模式
✅ 成功載入線上學習預測器
初始化 OnlinePredictor (v4 - 修正暖機邏輯)...
未找到初始模型，將在收到足夠數據後進行首次訓練。
正在初始化持久化的卡爾曼濾波器狀態...
初始化 OnlinePredictor (Y2 完整版 v2)...
未找到初始模型，將在收到足夠數據後進行首次訓練。 (./model\xgb_model_y2.json)
🏭 初始化 CFB#2 系統 (Y1 + Y2 預測)

正在生成 Y2/Y3 使用報告至 usage_report_truncated_600_y2.html...
報告已成功儲存至: usage_report_truncated_600_y2.html
---
🎉 使用截斷後的數據生成的新報告已儲存至：usage_report_truncated_600_y2.html
